In [2]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Question 1

In [5]:
len(documents)

72

In [6]:
len(files)

72

## Question 2

In [7]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

results = index.search(
    query="How does the agentic loop keep calling the model until it stops?",
    num_results=5
)

results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

## Question 3

In [8]:
import importlib
import rag_helper

importlib.reload(rag_helper)

rag = rag_helper.RAGBase(
    index=index,
    llm_client=openai_client
)

answer, usage = rag.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

print(answer)
print(usage)

The loop keeps calling the model with a `while True` loop, and after each response it checks whether the model returned any `function_call` items.

- If there are function calls, your code runs the tools, appends the tool outputs to `messages`, and loops again.
- If there are no function calls, it `break`s and stops.

So the exit condition is: **no function calls in the latest response**.
ResponseUsage(input_tokens=7126, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=92, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7218)


In [9]:
print(usage.input_tokens)

7126


## Question 4

In [10]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

len(chunks)

295

## Question 5

In [11]:
from gitsource import chunk_documents
from minsearch import Index

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

rag = rag_helper.RAGBase(
    index=chunk_index,
    llm_client=openai_client
)

answer, usage = rag.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

print(usage.input_tokens)

2309


In [12]:
7126 / usage.input_tokens

3.086184495452577

## Question 6

In [13]:
search_calls = 0

def search_course(query: str) -> str:
    """Search the lesson chunks."""
    
    global search_calls
    search_calls += 1

    results = chunk_index.search(
        query,
        num_results=5
    )

    return "\n\n".join(
        doc["content"]
        for doc in results
    )

In [18]:
search_calls = 0

class CourseTools:
    def search_course(self, query: str) -> str:
        """Search the course lesson chunks for information relevant to the query."""
        global search_calls
        search_calls += 1

        results = chunk_index.search(
            query,
            num_results=5
        )

        lines = []

        for doc in results:
            lines.append(f"FILE: {doc['filename']}")
            lines.append(doc["content"])
            lines.append("")

        return "\n".join(lines)

In [19]:
from toyaikit.tools import Tools

course_tools = CourseTools()

tools = Tools()
tools.add_tools(course_tools)

In [21]:
from toyaikit.chat import ChatAssistant, OpenAIClient, IPythonChatInterface
from toyaikit.tools import Tools

course_tools = CourseTools()

tools = Tools()
tools.add_tools(course_tools)

assistant = ChatAssistant(
    tools=tools,
    developer_prompt="""
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
""",
    chat_interface=IPythonChatInterface(),
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [30]:
search_calls = 0

result = assistant.runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    previous_messages=[],
)

print(result)
print("Search calls:", search_calls)

LoopResult(new_messages=[EasyInputMessage(content="\nYou're a course teaching assistant.\nAnswer the student's question using the search tool.\nMake multiple searches with different keywords before answering.\n", role='developer', phase=None, type=None), EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"agentic loop RAG plain RAG iterative retrieval course lesson"}', call_id='call_1XoE9UPSltOmNHZDEx2oxOgI', name='search_course', type='function_call', id='fc_023a7ee8968de67b006a2a477b4e4081a1bb0c1e34e3f8cf3e', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"agentic loop how it works retrieval generation tool use course"}', call_id='call_GWZFhOCL64JF01lPhpgB269M', name='search_course', type='function_call', id='fc_023a7ee8968de67b006a2a477b4e5481a194f7e78cb35edf5a', namespace=None, status='completed'), ResponseFunctionTool